# Study Bounce — Telescope-position bounce tests (v2)

**Author:** Aaron Roodman  
**Date Created:** 2026-05-06  
**Last Modified:** 2026-06-08  
**Status:** Draft  
**Keywords:** AOS, FAM, bounce test, BLOCK-T720, BLOCK-T724, double Zernike

## Description

BLOCK-T720 and BLOCK-T724 are *bounce* tests: take a FAM triplet at one
telescope orientation, move to a second orientation, take another
triplet, and repeat — alternating back and forth.

* **T720** bounces between Elevation = 70° and Elevation = 40°
  (Rotator near 0° throughout).
* **T724** bounces between Rotator = 0° and Rotator = 60°
  (Elevation near 70° throughout).

This notebook reads one or more `*_fits.parquet` files, identifies the
visits in each *setting* of each bounce, and builds the **paired-difference** Δ (comparison − reference): visits
are paired in time order within a `day_obs`, Δ is the median of the
per-pair differences, and the error is the scaled-MAD robust RMS of
those differences / √n_pairs — absorbing slow time variation without a
model.  The same paired Δ is computed for the Double-Zernike
coefficients, the OFC **v-modes** (`c_i = a_i/σ_i`), and the physical
**DOF** (50 DoF, n_keep=34, geom_mean norm, as in
`build_measured_intrinsic`).  Summary plots: ΔDZ / significance / pass
heat-maps over (k, j), DZ / v-mode / DOF vs ordinal-image plots, and a
5-panel DOF night-A-vs-night-B scatter.

**Output:** PDFs of bounce summary heatmaps, DZ / v-mode / DOF
vs-ordinal plots, and the DOF night-vs-night scatter, plus a
long-format parquet of per-(k, j) statistics.  Files in
`output/study_bounce/`.

**Based on:** the per-(k, j) tracking conventions used in the
`fit_params_resid_*` plots and `study_50dofLUT.ipynb`.  Designed to be
extended to more bounce tests with different telescope positions.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-05-06 | Aaron Roodman | Initial version: T720 (Elev 70/40) + T724 (Rot 0/60) heatmap summary. |
| 2026-06-08 | Aaron Roodman | Switched default input to the wep17_3_0 bin2x 20260418–20260531 fits parquet; added DZ_kj-vs-ordinal-image plots (standard marker scheme, day_obs dividers); factored the Δ-DZ analysis into run_bounce so it runs per-night as well as combined, and added a night-to-night Δ(Δ DZ_kj) difference heatmap (+pass map) with per-night errors in quadrature. |
| 2026-06-08 (v2) | Aaron Roodman | DZ-vs-ordinal plots now restricted to the bounce-test programs only, with day_obs annotated on the first panel.  Replaced the hard-to-read night-to-night Δ(Δ DZ) heatmaps with a per-night cross-comparison scatter: for each (k,j) passing the size+significance cut, plot night-A Δ DZ vs night-B Δ DZ with x/y SEM error bars and k,j labels, one panel per night pair (handles 3 nights -> 1v2, 1v3, 2v3). |
| 2026-06-08 (v3) | Aaron Roodman | Night-vs-night cross-scatter now one (larger, autoscaled) figure per night pair so the k,j labels are legible.  Term inclusion expanded to the union of the combined size+significance cut and each individual night's cut; added per-night pass heatmaps (green boxes) so it is clear which night drove each term's inclusion. |
| 2026-06-08 (v4) | Aaron Roodman | Significance cut now passes a cell if (|Δ| > pass_delta_threshold_um AND |Δ/σ| > 3.5) OR |Δ/σ| > pass_sigma_only_threshold (5σ, any size); raised pass_delta_threshold_um to 0.1 μm.  Added a zoomed (±0.1 μm) night-vs-night cross-scatter page per night pair alongside the autoscaled one. |
| 2026-06-08 (v5) | Aaron Roodman | Switched the Δ error method to a paired-difference estimator (median of time-ordered, within-night comp−ref pairs; err = scaled-MAD/√n_pairs) for DZ, v-modes and DOF — absorbs slow time variation without a model.  Extended the analysis beyond Double Zernikes to the OFC v-modes and physical DOF (50 DoF, n_keep=34, geom_mean norm, ported from build_measured_intrinsic): added v-mode-vs-ordinal and DOF-vs-ordinal plot pages and a 5-panel DOF night-A-vs-night-B scatter (hex pistons / decenters / tip-tilts, M1M3 + M2 bending) for T720 and T724. |
| 2026-06-09 | Aaron Roodman | Extracted the OFC SVD / DOF machinery (LABELS_50DOF, DOF_UNITS_50, DOF_GROUPS, recover_dof_per_visit, build_ofc_svd, project_dz_table) into the shared `code/ofc_svd.py` module; the notebook now imports it instead of carrying its own copy.  No change to results. |


## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [OFC v-mode / DOF setup](#svd)
6. [Bounce Δ Analysis (DZ + v-mode + DOF, combined + per-night)](#analysis)
7. [DZ_kj vs ordinal image number](#ordinal)
8. [v-mode & DOF vs ordinal image number](#vmdof-ordinal)
9. [Summary heatmaps (+ night-to-night repeatability)](#plots)
10. [DOF night-A vs night-B scatter](#dof-scatter)
11. [Output Tables](#output)


<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# ----- Input fit parquets (one or more) -----
# Multiple files are concatenated row-wise.  Bad-flagged visits are
# dropped before any statistics are computed.
fits_parquet_paths = [
    'output/aos_fam_danish_1_0_wep17_3_0_bin2x_20260418_20260531_fits.parquet',
]

# Fit prefix.
fit_prefix = 'z1toz6'

# Pupil-Noll j list.  None -> auto from the `nollIndices` column on
# the first fits parquet.
pupil_j_range = None
# Focal-plane Noll k list.
focal_k_range = range(1, 7)

# ----- Bounce specifications -----
# Each bounce has a `program` (a science_program value or list of
# values to filter to), a `reference` setting, and one or more
# `comparisons`.  A *setting* is a dict of selection criteria; only
# the keys that are present are applied.  Allowed keys (degrees /
# day_obs ints):
#     alt_range       -> (lo, hi) elevation range
#     rotator_range   -> (lo, hi)
#     day_obs_range   -> (lo, hi)
#     seq_num_range   -> (lo, hi)
#     mask            -> arbitrary callable: f(fit_table) -> bool array
#
# `program` is applied to every setting in that bounce so both the
# reference and the comparison only pull from the matching block —
# fixes the case where alt/rot windows accidentally include visits
# from BLOCK-T614 or BLOCK-T710_v2.
#
# The `delta` heatmap shows comparison.median - reference.median per
# (k, j), with errors from the per-setting MAD-based standard error of
# the median added in quadrature.
bounces = [
    {
        'name': 'T720_elevation',
        'description': 'Elevation 40 − 70 deg, rotator ≈ 0',
        'program': 'BLOCK-T720',
        'reference': {
            'label': 'Elev=70',
            'alt_range':     (67.0, 73.0),
            'rotator_range': (-3.0,  3.0),
        },
        'comparisons': [
            {
                'label': 'Elev=40',
                'alt_range':     (37.0, 43.0),
                'rotator_range': (-3.0,  3.0),
            },
        ],
    },
    {
        'name': 'T724_rotator',
        'description': 'Rotator 60 − 0 deg, elevation ≈ 70',
        'program': 'BLOCK-T724',
        'reference': {
            'label': 'Rot=0',
            'rotator_range': (-3.0,  3.0),
            'alt_range':     (67.0, 73.0),
        },
        'comparisons': [
            {
                'label': 'Rot=60',
                'rotator_range': (57.0, 63.0),
                'alt_range':     (67.0, 73.0),
            },
        ],
    },
]

# ----- Heatmap styling -----
# Color scale: per-bounce 95th-percentile of |Δ| if `vlim_um` is None,
# else use the explicit ±vlim_um for every heatmap.  Diverging colormap.
heatmap_vlim_um   = None
heatmap_cell_fontsize = 7
# Significance heatmap: Delta / err, capped at ±sig_vlim.
sig_vlim          = 5.0

# Pass/fail summary heatmap thresholds.  A (k, j) cell
# is marked as a *real* shift if BOTH
#     |Δ DZ_kj|    > pass_delta_threshold_um   (μm), AND
#     |Δ / σ|      > pass_nsigma_threshold     (unitless).
pass_nsigma_threshold   = 3.5
pass_delta_threshold_um = 0.1
# Second, OR'd criterion: a cell also passes if its significance alone
# exceeds this, regardless of |Δ| size.
pass_sigma_only_threshold = 5.0
# Fixed half-range (μm) for the zoomed night-vs-night cross-scatter
# page (inner region); None -> no zoom page.
cross_scatter_zoom_um = 0.1

# ----- DZ vs ordinal-image plots -----
# Pupil-j columns per page in the 'DZ_kj vs ordinal image number'
# pages (rows = focal k).  21 j -> 3 pages at 7.
ordinal_j_per_page = 7

# Per-night Delta-DZ: a night is a distinct day_obs.  A night is used
# in the per-night analysis when both the reference and comparison
# settings have at least this many visits on it.
night_min_visits = 3

# ----- Output -----
output_dir         = 'output/study_bounce'
output_pdf         = f'{output_dir}/study_bounce_summary.pdf'
output_pdf_ordinal = f'{output_dir}/study_bounce_dz_vs_ordinal.pdf'
output_table_path  = f'{output_dir}/study_bounce_kj_stats.parquet'

# ----- Degrees-of-freedom / v-mode decomposition -----
# The bounce Δ is also computed for the OFC v-modes and the physical
# DOF.  Setup mirrors build_measured_intrinsic (Path A): 50 DoF,
# n_keep=34, geom_mean normalization.  The DZ (k, j) slice used for the
# SVD is the same focal_k_range x pupil-j set used above.
n_dof  = 50
n_keep = 34
# OFC normalization-weights yaml; None -> auto from $TS_CONFIG_MTTCS_DIR
# (MTAOS/ofc/normalization_weights/range0.5_fwhm-0.15.yaml = geom_mean).
ofc_normalization_yaml = None

# vs-ordinal panel layout (one panel per v-mode / DOF).
vmode_ncols = 5
vmode_rows_per_page = 7
dof_ncols = 5
dof_rows_per_page = 10

# Extra outputs.
output_pdf_vmode_ordinal = f'{output_dir}/study_bounce_vmode_vs_ordinal.pdf'
output_pdf_dof_ordinal   = f'{output_dir}/study_bounce_dof_vs_ordinal.pdf'
output_pdf_dof_scatter   = f'{output_dir}/study_bounce_dof_night_scatter.pdf'


<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from astropy.table import QTable, vstack

sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, 'code')
from common.utils import setup_plotting

setup_plotting()

# Standard visit-marker scheme (elevation -> colour, rotator angle ->
# arrow direction, band -> edge colour).  Lives in intrinsics_lib,
# which imports the LSST stack at module load — available on RSP.
try:
    from intrinsics_lib import (classify_visit, visit_marker_style,
                                markers_legend_figure)
    _marker_ok = True
except Exception as _e:
    print(f'(intrinsics_lib marker scheme unavailable: '
          f'{type(_e).__name__}: {_e})')
    _marker_ok = False

print('Ready.')

<a id='functions'></a>
## 3. Helper Functions

In [ ]:
from common.zernike_names import (
    NOLL_NAMES, NOLL_FORMULAS, FOCAL_NAMES, PUPIL_NAMES,
)
from ofc_svd import (LABELS_50DOF, DOF_UNITS_50, DOF_GROUPS,
                     recover_dof_per_visit)
def _alt_to_deg(alt_arr):
    """Auto-detect radians vs degrees in `alt`."""
    a = np.asarray(alt_arr, dtype=float)
    if np.nanmax(np.abs(a)) < 2.0 * np.pi + 1e-3:
        return np.rad2deg(a)
    return a


def filter_visits(fit_table, *, alt_range=None, rotator_range=None,
                  day_obs_range=None, seq_num_range=None,
                  program=None, mask=None):
    """Build a boolean mask for visits matching every supplied criterion.

    All ranges are inclusive.  Missing keys are unrestricted.

    `program` matches the `science_program` column (case-sensitive
    exact match).  Pass a single string or a list/tuple of strings;
    the result is the union of all matches.  The fits parquet only
    started carrying `science_program` after the upstream
    `intrinsics_lib.merge_program_reason_to_visit_info` change — when
    the column is missing the program filter is silently skipped and a
    one-time warning is printed.
    """
    n = len(fit_table)
    keep = np.ones(n, dtype=bool)
    if alt_range is not None and 'alt' in fit_table.colnames:
        alt_deg = _alt_to_deg(fit_table['alt'])
        keep &= (alt_deg >= alt_range[0]) & (alt_deg <= alt_range[1])
    if rotator_range is not None and 'rotator_angle' in fit_table.colnames:
        rot = np.asarray(fit_table['rotator_angle'], dtype=float)
        keep &= (rot >= rotator_range[0]) & (rot <= rotator_range[1])
    if day_obs_range is not None and 'day_obs' in fit_table.colnames:
        d = np.asarray(fit_table['day_obs']).astype(int)
        keep &= (d >= int(day_obs_range[0])) & (d <= int(day_obs_range[1]))
    if seq_num_range is not None and 'seq_num' in fit_table.colnames:
        s = np.asarray(fit_table['seq_num']).astype(int)
        keep &= (s >= int(seq_num_range[0])) & (s <= int(seq_num_range[1]))
    if program is not None:
        if 'science_program' not in fit_table.colnames:
            if not getattr(filter_visits, '_warned_missing_program', False):
                print("  WARNING: 'science_program' not in fit_table — "
                      "the program filter is being ignored.  Re-run mktable "
                      "with the updated intrinsics_lib to populate it.")
                filter_visits._warned_missing_program = True
        else:
            sp = np.asarray(fit_table['science_program']).astype(str)
            if isinstance(program, (list, tuple, set)):
                prog_set = {str(p) for p in program}
                keep &= np.array([s in prog_set for s in sp])
            else:
                keep &= (sp == str(program))
    if mask is not None:
        keep &= np.asarray(mask, dtype=bool)
    return keep


def visit_list_str(fit_table, mask, max_per_day=20):
    """Pretty-print summary of which (day_obs, seq_num) visits survive a mask.

    Returns a string with one line per day_obs.  Truncates the seq_num
    list if it has more than `max_per_day` entries.
    """
    if not np.any(mask):
        return '    (no visits)'
    sub = fit_table[mask]
    dobs = np.asarray(sub['day_obs']).astype(int)
    snum = np.asarray(sub['seq_num']).astype(int)
    order = np.lexsort((snum, dobs))
    dobs = dobs[order]; snum = snum[order]
    lines = []
    for d in sorted(set(dobs.tolist())):
        s_list = sorted(snum[dobs == d].tolist())
        if len(s_list) > max_per_day:
            shown = (', '.join(str(x) for x in s_list[:max_per_day // 2])
                     + ', …, '
                     + ', '.join(str(x) for x in s_list[-max_per_day // 2:]))
            lines.append(f'    {d}: {len(s_list):3d} seq_num — '
                         f'[{shown}]')
        else:
            lines.append(f'    {d}: {len(s_list):3d} seq_num — '
                         f'{s_list}')
    return '\n'.join(lines)


def stats_per_kj(fit_table, mask, prefix, k_range, j_range):
    """Per-(k, j) median, robust RMS (1.4826*MAD), SEM of the median, n.

    SEM (standard error of the median) for normally distributed values
    is `1.2533 * sigma / sqrt(n)`; we use the MAD-based sigma estimate.
    """
    sub = fit_table[mask]
    out = {}
    for j in j_range:
        for k in k_range:
            col = f'{prefix}_z{j}_c{k}'
            if col not in sub.colnames:
                continue
            vals = np.asarray(sub[col], dtype=float)
            vals = vals[np.isfinite(vals)]
            n = int(len(vals))
            if n < 3:
                out[(int(k), int(j))] = {
                    'median': np.nan, 'sigma_mad': np.nan,
                    'sem': np.nan, 'n': n}
                continue
            med = float(np.median(vals))
            mad = float(np.median(np.abs(vals - med)))
            sigma_mad = 1.4826 * mad
            sem = 1.2533 * sigma_mad / np.sqrt(n)
            out[(int(k), int(j))] = {
                'median': med, 'sigma_mad': sigma_mad,
                'sem': sem, 'n': n}
    return out


def diff_stats(stats_comp, stats_ref):
    """Difference (comparison - reference) per (k, j) with quadrature errors."""
    out = {}
    keys = set(stats_comp.keys()) & set(stats_ref.keys())
    for kj in keys:
        a = stats_comp[kj]; b = stats_ref[kj]
        if not (np.isfinite(a['median']) and np.isfinite(b['median'])):
            out[kj] = {'delta': np.nan, 'err': np.nan,
                       'sig': np.nan,
                       'n_comp': a['n'], 'n_ref': b['n']}
            continue
        delta = a['median'] - b['median']
        err = float(np.sqrt(a['sem'] ** 2 + b['sem'] ** 2))
        sig = delta / err if err > 0 else np.nan
        out[kj] = {'delta': delta, 'err': err, 'sig': sig,
                   'n_comp': a['n'], 'n_ref': b['n']}
    return out


def _kj_to_array(stats_dict, k_list, j_list, key):
    """Pack one statistic into a (n_k, n_j) array for imshow."""
    Z = np.full((len(k_list), len(j_list)), np.nan)
    for (k, j), s in stats_dict.items():
        if k in k_list and j in j_list:
            Z[k_list.index(k), j_list.index(j)] = s.get(key, np.nan)
    return Z


def plot_kj_heatmap(stats, k_list, j_list, *, value_key='delta',
                    err_key='err', title='', cbar_label='',
                    cmap='RdBu_r', vlim=None, value_fmt='{:+.2f}',
                    err_fmt='±{:.2f}', cell_fontsize=7,
                    show_text=True):
    """Heatmap with k on rows, j on columns, value in colour, and
    optionally `value\n±err` in each cell.

    Returns the figure.
    """
    Z = _kj_to_array(stats, k_list, j_list, value_key)
    Errs = (_kj_to_array(stats, k_list, j_list, err_key)
            if err_key else None)
    if vlim is None:
        finite = Z[np.isfinite(Z)]
        vlim = (float(np.nanpercentile(np.abs(finite), 95))
                if finite.size else 1.0)
        vlim = max(vlim, 1e-4)

    nk, nj = len(k_list), len(j_list)
    fig, ax = plt.subplots(
        figsize=(max(8.0, 0.55 * nj + 1.5),
                 max(2.8, 0.65 * nk + 1.5)),
        layout='constrained')
    im = ax.imshow(Z, cmap=cmap, vmin=-vlim, vmax=vlim,
                   aspect='auto')
    ax.set_xticks(range(nj))
    ax.set_xticklabels([f'Z{j}' for j in j_list], fontsize=8)
    ax.set_yticks(range(nk))
    ax.set_yticklabels([str(k) for k in k_list])
    ax.set_xlabel('Pupil Zernike index j')
    ax.set_ylabel('Field index k')
    cb = plt.colorbar(im, ax=ax, shrink=0.85)
    cb.set_label(cbar_label)

    if show_text:
        for ri in range(nk):
            for ci in range(nj):
                v = Z[ri, ci]
                if not np.isfinite(v):
                    continue
                txt = value_fmt.format(v)
                if Errs is not None and np.isfinite(Errs[ri, ci]):
                    txt += '\n' + err_fmt.format(Errs[ri, ci])
                # Pick black or white text depending on cell brightness.
                color = ('white' if abs(v) > 0.55 * vlim else 'black')
                ax.text(ci, ri, txt, ha='center', va='center',
                        fontsize=cell_fontsize, color=color)

    if title:
        ax.set_title(title, fontsize=12)
    return fig


def to_long_df(stats, bounce_name, ref_label, comp_label,
                ref_stats, comp_stats, night='all'):
    """Long-format DataFrame for one bounce comparison."""
    rows = []
    for (k, j), d in sorted(stats.items()):
        r = ref_stats.get((k, j), {})
        c = comp_stats.get((k, j), {})
        rows.append({
            'bounce':      bounce_name,
            'night':       str(night),
            'reference':   ref_label,
            'comparison':  comp_label,
            'k': int(k), 'j': int(j),
            'n_ref':       int(r.get('n', 0)),
            'ref_median':  float(r.get('median', np.nan)),
            'ref_sigma':   float(r.get('sigma_mad', np.nan)),
            'ref_sem':     float(r.get('sem', np.nan)),
            'n_comp':      int(c.get('n', 0)),
            'comp_median': float(c.get('median', np.nan)),
            'comp_sigma':  float(c.get('sigma_mad', np.nan)),
            'comp_sem':    float(c.get('sem', np.nan)),
            'delta':       float(d.get('delta', np.nan)),
            'delta_err':   float(d.get('err', np.nan)),
            'significance': float(d.get('sig', np.nan)),
        })
    return pd.DataFrame(rows)


def plot_kj_pass_heatmap(deltas, k_list, j_list, *,
                         nsigma_threshold=3.5,
                         delta_threshold_um=0.01,
                         sigma_only_threshold=None,
                         title='',
                         pass_color='#1a936f',
                         cell_fontsize=7,
                         show_text=True):
    """Binary pass/fail heatmap over (k, j).

    A cell passes when BOTH ``|Δ| > delta_threshold_um`` and
    ``|Δ / σ| > nsigma_threshold``.  Passing cells get the single
    ``pass_color`` background and an annotation ``Δ\n±err``; failing
    cells stay white with no annotation.

    Returns the figure.
    """
    from matplotlib.colors import ListedColormap

    nk, nj = len(k_list), len(j_list)
    D = np.full((nk, nj), np.nan)
    E = np.full((nk, nj), np.nan)
    S = np.full((nk, nj), np.nan)
    for (k, j), d in deltas.items():
        if k in k_list and j in j_list:
            ri = k_list.index(k); ci = j_list.index(j)
            D[ri, ci] = d.get('delta', np.nan)
            E[ri, ci] = d.get('err',   np.nan)
            S[ri, ci] = d.get('sig',   np.nan)

    finite_m = np.isfinite(D) & np.isfinite(S)
    cutA = (np.abs(D) > delta_threshold_um) & (np.abs(S) > nsigma_threshold)
    cutB = ((np.abs(S) > sigma_only_threshold)
            if sigma_only_threshold is not None
            else np.zeros_like(cutA, dtype=bool))
    passes = finite_m & (cutA | cutB)
    n_pass = int(passes.sum())
    n_total = int(np.isfinite(D).sum())

    fig, ax = plt.subplots(
        figsize=(max(8.0, 0.55 * nj + 1.5),
                 max(2.8, 0.65 * nk + 1.5)),
        layout='constrained')
    cmap = ListedColormap(['white', pass_color])
    ax.imshow(passes.astype(int), cmap=cmap, vmin=0, vmax=1,
              aspect='auto')
    # Light grid lines between cells for readability.
    ax.set_xticks(np.arange(-0.5, nj, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, nk, 1), minor=True)
    ax.grid(which='minor', color='lightgray', linewidth=0.4, alpha=0.6)
    ax.tick_params(which='minor', bottom=False, left=False)

    ax.set_xticks(range(nj))
    ax.set_xticklabels([f'Z{j}' for j in j_list], fontsize=8)
    ax.set_yticks(range(nk))
    ax.set_yticklabels([str(k) for k in k_list])
    ax.set_xlabel('Pupil Zernike index j')
    ax.set_ylabel('Field index k')

    if show_text:
        for ri in range(nk):
            for ci in range(nj):
                if not passes[ri, ci]:
                    continue
                txt = f'{D[ri, ci]:+.3f}\n±{E[ri, ci]:.3f}'
                ax.text(ci, ri, txt, ha='center', va='center',
                        fontsize=cell_fontsize, color='white')

    base = title or 'Significant (k, j) terms'
    crit = (f'(|Δ| > {delta_threshold_um:g} μm AND '
            f'|Δ/σ| > {nsigma_threshold:g})')
    if sigma_only_threshold is not None:
        crit += f'  OR  |Δ/σ| > {sigma_only_threshold:g}'
    ax.set_title(f'{base}\nPass: {crit}    '
                 f'({n_pass} / {n_total} cells pass)', fontsize=11)
    return fig

def run_bounce(fit_table, b, prefix, k_list, j_list, day_obs=None):
    """Reference/comparison per-(k, j) stats and *paired* Δ for one bounce.

    The Δ is computed with the paired-difference method (see
    `form_pairs` / `paired_delta`): reference and comparison visits are
    paired in time order (never across a day_obs boundary), Δ is the
    median of the per-pair (comp − ref) differences, and the error is
    the scaled-MAD robust RMS of those differences / √n_pairs.  This
    absorbs slow time variation without modelling it.  `stats_per_kj`
    is still computed per setting for descriptive medians/RMS in the
    long table.  If `day_obs` is given, restrict to that single night.

    Returns {'ref_stats','ref_n','ref_mask',
             'comparisons': {label: {'comp_stats','comp_n','deltas',
                                     'comp_mask','pairs'}}}.
    """
    program = b.get('program')
    ref = b['reference']
    extra = {} if day_obs is None else {'day_obs_range': (int(day_obs), int(day_obs))}

    ref_kwargs = {k: v for k, v in ref.items() if k != 'label'}
    ref_kwargs.setdefault('program', program)
    ref_kwargs.update(extra)
    ref_mask = filter_visits(fit_table, **ref_kwargs)
    ref_stats = stats_per_kj(fit_table, ref_mask, prefix, k_list, j_list)

    comps = {}
    for comp in b['comparisons']:
        ck = {k: v for k, v in comp.items() if k != 'label'}
        ck.setdefault('program', program)
        ck.update(extra)
        cm = filter_visits(fit_table, **ck)
        cs = stats_per_kj(fit_table, cm, prefix, k_list, j_list)
        pairs = form_pairs(fit_table, ref_mask, cm)
        comps[comp['label']] = {
            'comp_stats': cs, 'comp_n': int(cm.sum()),
            'deltas': paired_deltas_kj(fit_table, prefix, k_list, j_list, pairs),
            'comp_mask': cm, 'pairs': pairs,
        }
    return {'ref_stats': ref_stats, 'ref_n': int(ref_mask.sum()),
            'ref_mask': ref_mask, 'comparisons': comps}

def bounce_nights(fit_table, b, prefix, k_list, j_list, min_visits=3):
    """Distinct day_obs nights where both ref and (each) comparison have
    >= min_visits, with per-night run_bounce results.

    Returns {night: run_bounce_result} for qualifying nights (sorted).
    """
    program = b.get('program')
    pmask = filter_visits(fit_table, program=program)
    if not np.any(pmask):
        return {}
    nights = sorted(set(np.asarray(fit_table['day_obs'])[pmask]
                        .astype(int).tolist()))
    out = {}
    for d in nights:
        rb = run_bounce(fit_table, b, prefix, k_list, j_list, day_obs=d)
        ok = rb['ref_n'] >= min_visits and all(
            c['comp_n'] >= min_visits for c in rb['comparisons'].values())
        if ok:
            out[d] = rb
    return out


def diff_of_deltas(deltas_a, deltas_b):
    """Difference of two Δ-DZ dicts (night A − night B) per (k, j), with
    quadrature-combined errors and significance."""
    out = {}
    for kj in set(deltas_a) & set(deltas_b):
        a, b = deltas_a[kj], deltas_b[kj]
        if not (np.isfinite(a.get('delta', np.nan))
                and np.isfinite(b.get('delta', np.nan))):
            out[kj] = {'delta': np.nan, 'err': np.nan, 'sig': np.nan}
            continue
        d = a['delta'] - b['delta']
        e = float(np.sqrt(a.get('err', np.nan) ** 2 + b.get('err', np.nan) ** 2))
        out[kj] = {'delta': d, 'err': e,
                   'sig': (d / e if e > 0 else np.nan)}
    return out


def plot_dz_vs_ordinal_pages(fit_table, prefix, k_list, j_list,
                             j_per_page=7):
    """Pages of DZ_kj vs ordinal image number (rows = focal k, cols =
    pupil j).  Points use the standard intrinsics_lib marker scheme
    (elevation -> colour, rotator angle -> arrow); dotted vertical lines
    mark day_obs changes.  Returns a list of figures.
    """
    dobs = np.asarray(fit_table['day_obs']).astype(int)
    snum = np.asarray(fit_table['seq_num']).astype(int)
    order = np.lexsort((snum, dobs))
    ft = fit_table[order]
    dobs = dobs[order]
    n = len(ft)
    ordinal = np.arange(n)
    alt = (_alt_to_deg(ft['alt']) if 'alt' in ft.colnames
           else np.full(n, np.nan))
    rot = (np.asarray(ft['rotator_angle'], dtype=float)
           if 'rotator_angle' in ft.colnames else np.full(n, np.nan))
    has_band = 'band' in ft.colnames
    band = np.asarray(ft['band']).astype(str) if has_band else None

    styles = []
    for i in range(n):
        if _marker_ok:
            cls = classify_visit(alt_deg=alt[i], rot_deg=rot[i],
                                 band=(band[i] if has_band else None))
            styles.append(visit_marker_style(elev=cls['elev'], rot=cls['rot'],
                                             band=cls['band'], base_size=4))
        else:
            styles.append(dict(marker='o', color='steelblue',
                               markersize=3, linestyle=''))

    changes = [i for i in range(1, n) if dobs[i] != dobs[i - 1]]
    day_labels = [(i, int(dobs[i])) for i in [0] + changes]

    figs = []
    for pg in range(0, len(j_list), j_per_page):
        jchunk = list(j_list[pg:pg + j_per_page])
        nrows, ncols = len(k_list), len(jchunk)
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(2.7 * ncols + 1.0, 1.8 * nrows + 1.0),
                                 layout='constrained', sharex=True,
                                 squeeze=False)
        for ri, k in enumerate(k_list):
            for ci, j in enumerate(jchunk):
                ax = axes[ri][ci]
                col = f'{prefix}_z{j}_c{k}'
                if col not in ft.colnames:
                    ax.set_visible(False)
                    continue
                y = np.asarray(ft[col], dtype=float)
                for i in range(n):
                    if np.isfinite(y[i]):
                        ax.plot(ordinal[i], y[i], **styles[i])
                for ch in changes:
                    ax.axvline(ch - 0.5, color='gray', ls=':', lw=0.6,
                               alpha=0.7)
                ax.axhline(0, color='k', lw=0.4, alpha=0.4)
                if ri == 0:
                    ax.set_title(f'Z{j}', fontsize=8)
                if ci == 0:
                    ax.set_ylabel(f'k={k}', fontsize=8)
                ax.tick_params(labelsize=6)
        # Annotate the day_obs at each day-start on the first panel so
        # the ordinal axis can be read back to calendar nights.
        ax0 = axes[0][0]
        for (start_i, day) in day_labels:
            ax0.text(start_i, 0.98, str(day),
                     transform=ax0.get_xaxis_transform(),
                     rotation=90, va='top', ha='left',
                     fontsize=5, color='dimgray', alpha=0.9)
        for ax in axes[-1]:
            ax.set_xlabel('ordinal image #', fontsize=7)
        fig.suptitle(f'{prefix}  DZ_kj vs ordinal image number  '
                     f'(pupil j {jchunk[0]}..{jchunk[-1]})  '
                     f'— dotted lines = day_obs change', fontsize=12)
        figs.append(fig)
    return figs

def passing_terms(deltas, delta_th, nsigma_th, sigma_only_th=None):
    """Set of (k, j) passing the significance cut.

    A term passes if  (|Δ| > delta_th AND |Δ/σ| > nsigma_th)  OR
    (sigma_only_th is not None AND |Δ/σ| > sigma_only_th).
    """
    out = set()
    for (k, j), d in deltas.items():
        dl = d.get('delta', np.nan); sg = d.get('sig', np.nan)
        if not (np.isfinite(dl) and np.isfinite(sg)):
            continue
        cutA = abs(dl) > delta_th and abs(sg) > nsigma_th
        cutB = sigma_only_th is not None and abs(sg) > sigma_only_th
        if cutA or cutB:
            out.add((k, j))
    return out


def plot_night_cross_scatter(deltas_by_night, passing_kj, title_root='',
                             zoom_lim_um=None):
    """Night-A vs Night-B Δ DZ_kj cross-comparison, one figure per night
    pair (plus a zoomed inner-region page when `zoom_lim_um` is given).

    Each passing (k, j) is an errorbar point (per-night SEM as x/y
    errors) annotated with k,j; a y = x line is drawn.  The full page
    autoscales to the data (+ errors); the zoom page fixes the axes to
    ±`zoom_lim_um`.  Returns a list of figures
    (full[, zoom] per pair; empty if < 2 nights or no passing terms).
    """
    import itertools
    nights = sorted(deltas_by_night.keys())
    figs = []
    if len(nights) < 2 or not passing_kj:
        return figs

    def _make(A, B, xs, ys, xe, ye, labs, lim):
        fig, ax = plt.subplots(figsize=(9, 9), layout='constrained')
        ax.errorbar(xs, ys, xerr=xe, yerr=ye, fmt='o', ms=6,
                    color='steelblue', ecolor='gray', elinewidth=0.9,
                    capsize=2, alpha=0.85, zorder=3)
        for x, y, l in zip(xs, ys, labs):
            ax.annotate(l, (x, y), textcoords='offset points',
                        xytext=(5, 5), fontsize=8, color='black')
        if lim is None:
            lo = float(min((xs - xe).min(), (ys - ye).min()))
            hi = float(max((xs + xe).max(), (ys + ye).max()))
            pad = 0.10 * max(hi - lo, 1e-3)
            lo -= pad; hi += pad
            ztag = ''
        else:
            lo, hi = -float(lim), float(lim)
            ztag = f'  (zoom ±{lim:g} μm)'
        ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8, alpha=0.6, zorder=1)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.set_aspect('equal')
        ax.axhline(0, color='gray', lw=0.4)
        ax.axvline(0, color='gray', lw=0.4)
        ax.set_xlabel(f'Δ DZ_kj  night {A} [μm]', fontsize=11)
        ax.set_ylabel(f'Δ DZ_kj  night {B} [μm]', fontsize=11)
        ax.grid(alpha=0.3)
        ax.set_title(f'{title_root}\nnight {A} vs night {B}  '
                     f'({len(xs)} terms){ztag}', fontsize=12)
        return fig

    for (A, B) in itertools.combinations(nights, 2):
        da, db = deltas_by_night[A], deltas_by_night[B]
        xs, ys, xe, ye, labs = [], [], [], [], []
        for (k, j) in passing_kj:
            a = da.get((k, j)); b = db.get((k, j))
            if a is None or b is None:
                continue
            if np.isfinite(a.get('delta', np.nan)) and np.isfinite(b.get('delta', np.nan)):
                xs.append(a['delta']); ys.append(b['delta'])
                xe.append(a.get('err', np.nan)); ye.append(b.get('err', np.nan))
                labs.append(f'{k},{j}')
        if not xs:
            continue
        xs = np.array(xs); ys = np.array(ys)
        xe = np.nan_to_num(np.array(xe)); ye = np.nan_to_num(np.array(ye))
        figs.append(_make(A, B, xs, ys, xe, ye, labs, None))
        if zoom_lim_um is not None:
            figs.append(_make(A, B, xs, ys, xe, ye, labs, zoom_lim_um))
    return figs



# ==================================================================
# Paired-difference Δ engine (time-aware, model-free)
# ==================================================================
def form_pairs(fit_table, ref_mask, comp_mask):
    """Time-ordered, non-overlapping (reference, comparison) visit pairs.

    Walks all reference+comparison visits in (day_obs, seq_num) order
    and greedily pairs each visit with the nearest following visit of
    the opposite setting, never crossing a day_obs boundary.  A run of
    same-setting visits keeps only the most recent unpaired one.
    Returns a list of (ref_row, comp_row) integer row indices into
    `fit_table`.
    """
    n = len(fit_table)
    setting = np.zeros(n, dtype=int)
    setting[np.asarray(ref_mask, bool)] = -1
    setting[np.asarray(comp_mask, bool)] = 1          # comp wins any overlap
    active = np.nonzero(setting != 0)[0]
    if active.size == 0:
        return []
    dobs = np.asarray(fit_table['day_obs']).astype(int)
    snum = np.asarray(fit_table['seq_num']).astype(int)
    order = active[np.lexsort((snum[active], dobs[active]))]
    pairs = []
    pend = None
    for i in order:
        if (pend is not None and dobs[i] == dobs[pend]
                and setting[i] == -setting[pend]):
            r = pend if setting[pend] == -1 else i
            c = i if setting[i] == 1 else pend
            pairs.append((int(r), int(c)))
            pend = None
        else:
            pend = i
    return pairs


def paired_delta(values, pairs):
    """Paired-difference Δ for one quantity over (ref, comp) pairs.

    Δ = median(comp − ref) over pairs; err = (1.4826·MAD of the per-pair
    differences) / √n_pairs; sig = Δ / err.  Returns
    {delta, err, sig, n}  (n = number of finite pairs).
    """
    v = np.asarray(values, dtype=float)
    if not pairs:
        return {'delta': np.nan, 'err': np.nan, 'sig': np.nan, 'n': 0}
    diffs = np.array([v[c] - v[r] for (r, c) in pairs], dtype=float)
    diffs = diffs[np.isfinite(diffs)]
    n = int(diffs.size)
    if n < 1:
        return {'delta': np.nan, 'err': np.nan, 'sig': np.nan, 'n': 0}
    delta = float(np.median(diffs))
    sigma_mad = 1.4826 * float(np.median(np.abs(diffs - delta)))
    err = sigma_mad / np.sqrt(n)
    sig = delta / err if err > 0 else np.nan
    return {'delta': delta, 'err': float(err), 'sig': sig, 'n': n}


def paired_deltas_kj(fit_table, prefix, k_list, j_list, pairs):
    """Paired Δ per (k, j) for the DZ coefficients.  {(k, j): {...}}."""
    out = {}
    for j in j_list:
        for k in k_list:
            col = f'{prefix}_z{j}_c{k}'
            if col not in fit_table.colnames:
                continue
            out[(int(k), int(j))] = paired_delta(
                np.asarray(fit_table[col], dtype=float), pairs)
    return out


def paired_deltas_matrix(value_matrix, pairs, keys=None):
    """Paired Δ for each column of `value_matrix` (n_visits, n_q),
    aligned row-for-row to the fit_table the pairs index into.
    Returns {key: {delta,err,sig,n}} with key = keys[i] or int i."""
    M = np.asarray(value_matrix, dtype=float)
    nq = M.shape[1]
    ks = list(range(nq)) if keys is None else list(keys)
    return {ks[i]: paired_delta(M[:, i], pairs) for i in range(nq)}


# OFC v-mode / DOF recovery (LABELS_50DOF, DOF_UNITS_50,
# recover_dof_per_visit) now live in code/ofc_svd.py — imported above.


# ==================================================================
# Generic <quantity> vs ordinal-image plotting (marker scheme shared)
# ==================================================================
def _ordinal_setup(fit_table, base_size=4):
    """Time-sort a fit_table and build the shared per-visit marker
    styles, day_obs change indices and day labels for vs-ordinal plots.
    Returns a dict with order/ft/n/ordinal/styles/changes/day_labels."""
    dobs = np.asarray(fit_table['day_obs']).astype(int)
    snum = np.asarray(fit_table['seq_num']).astype(int)
    order = np.lexsort((snum, dobs))
    ft = fit_table[order]
    dobs = dobs[order]
    n = len(ft)
    alt = (_alt_to_deg(ft['alt']) if 'alt' in ft.colnames
           else np.full(n, np.nan))
    rot = (np.asarray(ft['rotator_angle'], dtype=float)
           if 'rotator_angle' in ft.colnames else np.full(n, np.nan))
    has_band = 'band' in ft.colnames
    band = np.asarray(ft['band']).astype(str) if has_band else None
    styles = []
    for i in range(n):
        if _marker_ok:
            cls = classify_visit(alt_deg=alt[i], rot_deg=rot[i],
                                 band=(band[i] if has_band else None))
            styles.append(visit_marker_style(
                elev=cls['elev'], rot=cls['rot'], band=cls['band'],
                base_size=base_size))
        else:
            styles.append(dict(marker='o', color='steelblue',
                               markersize=3, linestyle=''))
    changes = [i for i in range(1, n) if dobs[i] != dobs[i - 1]]
    day_labels = [(i, int(dobs[i])) for i in [0] + changes]
    return {'order': order, 'ft': ft, 'n': n, 'ordinal': np.arange(n),
            'styles': styles, 'changes': changes, 'day_labels': day_labels}


def plot_values_vs_ordinal_pages(fit_table, value_matrix, labels,
                                 units=None, title_root='', ncols=5,
                                 rows_per_page=7):
    """Pages of <quantity> vs ordinal image number, one panel per column
    of `value_matrix` (aligned row-for-row to `fit_table`).  Standard
    marker scheme; dotted day_obs lines; day_obs annotated on the first
    panel.  Returns a list of figures.
    """
    s = _ordinal_setup(fit_table)
    order, ordinal, styles = s['order'], s['ordinal'], s['styles']
    changes, day_labels, n = s['changes'], s['day_labels'], s['n']
    M = np.asarray(value_matrix, dtype=float)[order]
    nq = M.shape[1]
    per_page = ncols * rows_per_page
    figs = []
    for pg in range(0, nq, per_page):
        qs = list(range(pg, min(pg + per_page, nq)))
        nrows = int(np.ceil(len(qs) / ncols))
        fig, axes = plt.subplots(
            nrows, ncols, figsize=(2.7 * ncols + 1.0, 1.7 * nrows + 1.0),
            layout='constrained', sharex=True, squeeze=False)
        for cell, q in enumerate(qs):
            ax = axes[cell // ncols][cell % ncols]
            y = M[:, q]
            for i in range(n):
                if np.isfinite(y[i]):
                    ax.plot(ordinal[i], y[i], **styles[i])
            for ch in changes:
                ax.axvline(ch - 0.5, color='gray', ls=':', lw=0.6, alpha=0.7)
            ax.axhline(0, color='k', lw=0.4, alpha=0.4)
            lab = labels[q] if units is None else f'{labels[q]} [{units[q]}]'
            ax.set_title(lab, fontsize=8)
            ax.tick_params(labelsize=6)
        for cell in range(len(qs), nrows * ncols):
            axes[cell // ncols][cell % ncols].set_visible(False)
        ax0 = axes[0][0]
        for (start_i, day) in day_labels:
            ax0.text(start_i, 0.98, str(day),
                     transform=ax0.get_xaxis_transform(), rotation=90,
                     va='top', ha='left', fontsize=5, color='dimgray',
                     alpha=0.9)
        for c in range(ncols):
            axes[nrows - 1][c].set_xlabel('ordinal image #', fontsize=7)
        fig.suptitle(f'{title_root}  (panels {qs[0]}..{qs[-1]})  '
                     f'— dotted lines = day_obs change', fontsize=12)
        figs.append(fig)
    return figs


# ==================================================================
# DOF night-A vs night-B 5-panel scatter
# ==================================================================
DOF_PANELS = [
    ('Cam & M2 Hex piston',   DOF_GROUPS['hex_piston']),
    ('Cam & M2 Hex decenter', DOF_GROUPS['hex_decenter']),
    ('Cam & M2 Hex tip/tilt', DOF_GROUPS['hex_tiptilt']),
    ('M1M3 bending modes',    DOF_GROUPS['m1m3_bending']),
    ('M2 bending modes',      DOF_GROUPS['m2_bending']),
]


def plot_dof_night_scatter(dof_deltas_by_night, labels, units=None,
                           title_root='', night_pair=None):
    """5-panel (one page) night-A vs night-B scatter of the physical
    DOF Δ values: Cam/M2 hex pistons, hex decenters, hex tip/tilts,
    M1M3 bending (20), M2 bending (20) — all 50 DOF.  Each point is an
    errorbar (paired-Δ err per night) annotated with its DOF label; a
    y = x line is drawn per panel.  One figure per night pair (or just
    `night_pair` if given).  Returns a list of figures.
    """
    import itertools
    nights = sorted(dof_deltas_by_night.keys())
    if len(nights) < 2:
        return []
    night_pairs = ([tuple(night_pair)] if night_pair is not None
                   else list(itertools.combinations(nights, 2)))
    figs = []
    for (A, B) in night_pairs:
        da = dof_deltas_by_night.get(A)
        db = dof_deltas_by_night.get(B)
        if da is None or db is None:
            continue
        fig, axes = plt.subplots(1, 5, figsize=(24, 5.2),
                                 layout='constrained')
        for pi, (ptitle, qs) in enumerate(DOF_PANELS):
            ax = axes[pi]
            xs, ys, xe, ye, labs = [], [], [], [], []
            for q in qs:
                a = da.get(q); bb = db.get(q)
                if a is None or bb is None:
                    continue
                if (np.isfinite(a.get('delta', np.nan))
                        and np.isfinite(bb.get('delta', np.nan))):
                    xs.append(a['delta']); ys.append(bb['delta'])
                    xe.append(a.get('err', np.nan))
                    ye.append(bb.get('err', np.nan))
                    labs.append(labels[q])
            if xs:
                xs = np.array(xs); ys = np.array(ys)
                xe = np.nan_to_num(np.array(xe))
                ye = np.nan_to_num(np.array(ye))
                ax.errorbar(xs, ys, xerr=xe, yerr=ye, fmt='o', ms=5,
                            color='steelblue', ecolor='gray',
                            elinewidth=0.8, capsize=2, alpha=0.85, zorder=3)
                for x, y, l in zip(xs, ys, labs):
                    ax.annotate(l, (x, y), textcoords='offset points',
                                xytext=(4, 4), fontsize=6, color='black')
                lo = float(min((xs - xe).min(), (ys - ye).min()))
                hi = float(max((xs + xe).max(), (ys + ye).max()))
                pad = 0.10 * max(hi - lo, 1e-6)
                lo -= pad; hi += pad
                ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8, alpha=0.6,
                        zorder=1)
                ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
            unit = (f' [{units[qs[0]]}]' if units is not None else '')
            ax.axhline(0, color='gray', lw=0.4)
            ax.axvline(0, color='gray', lw=0.4)
            ax.set_aspect('equal', adjustable='datalim')
            ax.set_xlabel(f'night {A}{unit}', fontsize=9)
            ax.set_ylabel(f'night {B}{unit}', fontsize=9)
            ax.set_title(ptitle, fontsize=10)
            ax.grid(alpha=0.3)
        fig.suptitle(f'{title_root}\nDOF Δ:  night {A} vs night {B}',
                     fontsize=13)
        figs.append(fig)
    return figs

<a id='data'></a>
## 4. Data Access

In [ ]:
tables = []
for p in fits_parquet_paths:
    pp = Path(p)
    if not pp.exists():
        raise FileNotFoundError(pp)
    t = QTable.read(str(pp))
    print(f'Loaded {pp.name}: {len(t)} rows, {len(t.colnames)} cols')
    tables.append(t)
fit_table = vstack(tables, metadata_conflicts='silent') if len(tables) > 1 else tables[0]
print(f'\nCombined: {len(fit_table)} rows')

# Drop bad-fit visits.
for bf in (f'{fit_prefix}_bad_fit', 'bad_fit'):
    if bf in fit_table.colnames:
        bad = np.asarray(fit_table[bf]).astype(bool)
        n_bad = int(bad.sum())
        if n_bad > 0:
            print(f'Dropping {n_bad} bad-flagged visits ({bf})')
        fit_table = fit_table[~bad]
        break

# Resolve pupil-Noll j list.
if pupil_j_range is None:
    if 'nollIndices' not in fit_table.colnames:
        raise ValueError(
            "fits parquet has no 'nollIndices' column; set pupil_j_range "
            "explicitly in the parameters cell.")
    iZs = [int(j) for j in np.asarray(fit_table['nollIndices'][0]).tolist()]
else:
    iZs = list(pupil_j_range)
k_list = list(focal_k_range)
print(f'\nPupil Noll j (n={len(iZs)}): {iZs}')
print(f'Focal Noll k (n={len(k_list)}): {k_list}')

# Quick orientation: alt / rotator distribution to confirm the
# bounce-test settings actually live in the data.
if 'alt' in fit_table.colnames:
    alt_deg = _alt_to_deg(fit_table['alt'])
    print(f'\nalt: {np.nanmin(alt_deg):.1f} .. {np.nanmax(alt_deg):.1f} deg, '
          f'mean={np.nanmean(alt_deg):.1f}')
if 'rotator_angle' in fit_table.colnames:
    rot = np.asarray(fit_table['rotator_angle'], dtype=float)
    print(f'rotator_angle: {np.nanmin(rot):.1f} .. {np.nanmax(rot):.1f} deg')

<a id='svd'></a>
## 5. OFC v-mode / DOF setup

Build the OFC sensitivity-matrix SVD for the chosen focal-`k` × pupil-`j`
slice (right-multiplied by the geom_mean normalization), keep `n_keep`
U-modes, and project every visit's DZ vector onto the v-modes
(`c_i = a_i/σ_i`) and physical DOF (`recover_dof_per_visit`).  Mirrors
`build_measured_intrinsic` Path A.  Requires `lsst.ts.ofc` +
`$TS_CONFIG_MTTCS_DIR` (RSP); the v-mode / DOF sections are skipped
gracefully when unavailable.

In [ ]:
# OFC SVD + per-visit v-mode / DOF projection (see code/ofc_svd.py)
C_all = DOF_all = None
U_eff = V = Sigma = normalization_weights = kj_grid = None
n_keep_eff = 0
vmode_labels = []
ofc_svd = None

try:
    from ofc_svd import build_ofc_svd, project_dz_table
    ofc_svd = build_ofc_svd(
        iZs, int(min(k_list)), int(max(k_list)), n_keep,
        ofc_normalization_yaml=ofc_normalization_yaml)
    _svd_ok = True
except Exception as _e:
    print(f'(OFC SVD unavailable [{type(_e).__name__}: {_e}]; '
          f'v-mode / DOF sections skipped)')
    _svd_ok = False

if _svd_ok:
    # Expose the pieces downstream cells expect.
    U_eff = ofc_svd.U_eff
    V = ofc_svd.V
    Sigma = ofc_svd.Sigma
    normalization_weights = ofc_svd.normalization_weights
    kj_grid = ofc_svd.kj_grid
    n_keep_eff = ofc_svd.n_keep_eff
    vmode_labels = ofc_svd.vmode_labels
    print(f'OFC SVD: n_keep={n_keep_eff}  (k={ofc_svd.k_min}..{ofc_svd.k_max}, '
          f'{len(ofc_svd.iZs)} pupil-j, {ofc_svd.n_dof} DoF)')

    # Project every visit's raw DZ vector onto v-modes (c_i = a_i/sigma_i)
    # and physical DOF.
    C_all, DOF_all, A_modes, W = project_dz_table(fit_table, fit_prefix, ofc_svd)
    n_missing = int(np.isnan(W).any(axis=1).sum())
    if n_missing:
        print(f'  ({n_missing}/{len(fit_table)} visits have >=1 NaN DZ term; '
              f'treated as 0 in the projection)')
    print(f'Projected {len(fit_table)} visits -> C_all{C_all.shape}, '
          f'DOF_all{DOF_all.shape}')


<a id='analysis'></a>
## 6. Bounce Analysis

For each bounce in `bounces`:

1. Build masks for the reference and comparison settings.
2. Pair reference/comparison visits in time order (within a day_obs),
   and compute the **paired Δ**: Δ = median(comp − ref) over pairs,
   err = (1.4826·MAD of the per-pair differences) / √n_pairs.  This
   absorbs slow time variation without modelling it.
3. The same paired Δ is computed for the OFC **v-modes** (`c_i`) and the
   physical **DOF** (50), per comparison and per night.

All results land in `bounce_results`, indexed by bounce name and
comparison label.

In [ ]:
bounce_results = {}
long_dfs = []

if 'science_program' in fit_table.colnames:
    from collections import Counter
    sp = np.asarray(fit_table['science_program']).astype(str)
    print('science_program in fit_table:')
    for kk, nn in sorted(Counter(sp).items(), key=lambda kv: -kv[1]):
        print(f'  {kk!r}: {nn}')

for b in bounces:
    name = b['name']
    descr = b.get('description', '')
    combined = run_bounce(fit_table, b, fit_prefix, k_list, iZs, day_obs=None)
    nights = bounce_nights(fit_table, b, fit_prefix, k_list, iZs,
                           min_visits=night_min_visits)

    print(f'\n=== {name} ({descr}) ===')
    print(f'  program filter: {b.get("program")!r}')
    print(f'  reference "{b["reference"]["label"]}": '
          f'n_visits = {combined["ref_n"]} (all nights)')
    print(visit_list_str(fit_table, combined['ref_mask']))
    print(f'  qualifying nights (>= {night_min_visits} visits both '
          f'settings): {sorted(nights.keys())}')

    br = {'description': descr,
          'reference_label': b['reference']['label'],
          'reference_stats': combined['ref_stats'],
          'reference_n': combined['ref_n'],
          'comparisons': {}}

    for comp in b['comparisons']:
        label = comp['label']
        cblock = combined['comparisons'][label]
        pairs_all = cblock['pairs']
        deltas_by_night = {d: nights[d]['comparisons'][label]['deltas']
                           for d in nights}
        compn_by_night = {d: nights[d]['comparisons'][label]['comp_n']
                          for d in nights}
        refn_by_night = {d: nights[d]['ref_n'] for d in nights}

        night_pair = night_diff = None
        if len(deltas_by_night) >= 2:
            two = sorted(deltas_by_night.keys(),
                         key=lambda d: -compn_by_night[d])[:2]
            two = sorted(two)            # chronological order
            night_pair = (int(two[0]), int(two[1]))
            night_diff = diff_of_deltas(deltas_by_night[two[0]],
                                        deltas_by_night[two[1]])

        # ---- v-mode and physical-DOF paired deltas (combined + per night)
        vmode_deltas = dof_deltas = None
        vmode_deltas_by_night = dof_deltas_by_night = {}
        if _svd_ok and C_all is not None:
            vmode_deltas = paired_deltas_matrix(C_all, pairs_all)
            dof_deltas = paired_deltas_matrix(DOF_all, pairs_all)
            vmode_deltas_by_night = {
                d: paired_deltas_matrix(
                    C_all, nights[d]['comparisons'][label]['pairs'])
                for d in nights}
            dof_deltas_by_night = {
                d: paired_deltas_matrix(
                    DOF_all, nights[d]['comparisons'][label]['pairs'])
                for d in nights}

        br['comparisons'][label] = {
            'comp_stats': cblock['comp_stats'],
            'comp_n': cblock['comp_n'],
            'deltas': cblock['deltas'],
            'pairs': pairs_all,
            'n_pairs': len(pairs_all),
            'deltas_by_night': deltas_by_night,
            'compn_by_night': compn_by_night,
            'refn_by_night': refn_by_night,
            'night_pair': night_pair,
            'night_diff': night_diff,
            'vmode_deltas': vmode_deltas,
            'dof_deltas': dof_deltas,
            'vmode_deltas_by_night': vmode_deltas_by_night,
            'dof_deltas_by_night': dof_deltas_by_night,
        }

        print(f'  comparison "{label}": n_visits = {cblock["comp_n"]} '
              f'(all nights), {len(pairs_all)} time-ordered pairs')
        print(visit_list_str(fit_table, cblock['comp_mask']))
        if night_pair:
            print(f'    night-difference pair: {night_pair[0]} - '
                  f'{night_pair[1]}  (comp n = '
                  f'{compn_by_night[night_pair[0]]}, '
                  f'{compn_by_night[night_pair[1]]})')

        ranked = sorted([(kj, d) for kj, d in cblock['deltas'].items()
                         if np.isfinite(d.get('sig', np.nan))],
                        key=lambda kvd: -abs(kvd[1]['sig']))[:5]
        for (k, j), d in ranked:
            print(f'    top |sig|: (k={k}, j={j})  '
                  f'D={d["delta"]:+.3f} +/- {d["err"]:.3f}  '
                  f'sig={d["sig"]:+.1f}  (n_pairs={d["n"]})')
        if dof_deltas is not None:
            dranked = sorted([(q, d) for q, d in dof_deltas.items()
                             if np.isfinite(d.get('sig', np.nan))],
                            key=lambda kvd: -abs(kvd[1]['sig']))[:5]
            for q, d in dranked:
                print(f'    top |sig| DOF: {LABELS_50DOF[q]:7s}  '
                      f'D={d["delta"]:+.4g} {DOF_UNITS_50[q]}  '
                      f'sig={d["sig"]:+.1f}')

        # long-format rows: combined ('all') + per night
        long_dfs.append(to_long_df(
            cblock['deltas'], name, br['reference_label'], label,
            combined['ref_stats'], cblock['comp_stats'], night='all'))
        for d in deltas_by_night:
            long_dfs.append(to_long_df(
                deltas_by_night[d], name, br['reference_label'], label,
                nights[d]['ref_stats'],
                nights[d]['comparisons'][label]['comp_stats'], night=d))

    bounce_results[name] = br

df_kj = (pd.concat(long_dfs, ignore_index=True)
         if long_dfs else pd.DataFrame())
print(f'\nLong-format table: {len(df_kj)} rows (combined + per-night)')


<a id='ordinal'></a>
## DZ_kj vs ordinal image number

Per-(k, j) Double-Zernike value vs ordinal image number (visits sorted
by day_obs, seq_num).  Points use the standard marker scheme —
elevation sets the colour and rotator angle the arrow direction; a
dotted vertical line marks each change of `day_obs`.  Streamed to
`study_bounce_dz_vs_ordinal.pdf` (rows = focal k, `ordinal_j_per_page` pupil-j per page).

In [ ]:
Path(output_dir).mkdir(parents=True, exist_ok=True)
# Restrict to the bounce-test programs only (drop unrelated visits in
# the parquet) before plotting DZ vs ordinal image number.
_bounce_progs = []
for _b in bounces:
    _p = _b.get('program')
    if isinstance(_p, (list, tuple, set)):
        _bounce_progs += [str(x) for x in _p]
    elif _p is not None:
        _bounce_progs.append(str(_p))
_bounce_progs = sorted(set(_bounce_progs))
_omask = (filter_visits(fit_table, program=_bounce_progs)
          if _bounce_progs else np.ones(len(fit_table), dtype=bool))
ft_ord = fit_table[_omask]
print(f'DZ-vs-ordinal: {len(ft_ord)}/{len(fit_table)} visits in bounce '
      f'programs {_bounce_progs}')
ord_figs = plot_dz_vs_ordinal_pages(ft_ord, fit_prefix, k_list, iZs,
                                    j_per_page=ordinal_j_per_page)
with PdfPages(output_pdf_ordinal) as pdf:
    if _marker_ok:
        _leg = markers_legend_figure(show_iter_distinction=False)
        pdf.savefig(_leg, bbox_inches='tight')
        plt.show()
        plt.close(_leg)
    for _fnum, _fig in enumerate(ord_figs):
        pdf.savefig(_fig, bbox_inches='tight')
        if _fnum == 0:
            plt.show()
        plt.close(_fig)
print(f'Wrote {len(ord_figs)} DZ-vs-ordinal page(s) (+legend) to '
      f'{output_pdf_ordinal}')


<a id='vmdof-ordinal'></a>
## 7. v-mode & DOF vs ordinal image number

Per-visit OFC v-mode amplitudes (`c_i = a_i/σ_i`, `n_keep` of them) and
the recovered physical DOF (50, mixed μm / arcsec) vs ordinal image
number, same marker scheme and day_obs dividers as the DZ pages.
Restricted to the bounce-test programs.  Streamed to
`study_bounce_vmode_vs_ordinal.pdf` and
`study_bounce_dof_vs_ordinal.pdf`.

In [ ]:
# v-mode and DOF vs ordinal image number (bounce programs only)
if _svd_ok and C_all is not None:
    C_ord = C_all[_omask]
    DOF_ord = DOF_all[_omask]

    vmode_figs = plot_values_vs_ordinal_pages(
        ft_ord, C_ord, vmode_labels, units=None,
        title_root='OFC v-mode amplitude c_i',
        ncols=vmode_ncols, rows_per_page=vmode_rows_per_page)
    with PdfPages(output_pdf_vmode_ordinal) as pdf:
        if _marker_ok:
            _leg = markers_legend_figure(show_iter_distinction=False)
            pdf.savefig(_leg, bbox_inches='tight'); plt.close(_leg)
        for _i, _f in enumerate(vmode_figs):
            pdf.savefig(_f, bbox_inches='tight')
            if _i == 0:
                plt.show()
            plt.close(_f)
    print(f'Wrote {len(vmode_figs)} v-mode page(s) to '
          f'{output_pdf_vmode_ordinal}')

    dof_figs = plot_values_vs_ordinal_pages(
        ft_ord, DOF_ord, LABELS_50DOF, units=DOF_UNITS_50,
        title_root='Physical DOF', ncols=dof_ncols,
        rows_per_page=dof_rows_per_page)
    with PdfPages(output_pdf_dof_ordinal) as pdf:
        if _marker_ok:
            _leg = markers_legend_figure(show_iter_distinction=False)
            pdf.savefig(_leg, bbox_inches='tight'); plt.close(_leg)
        for _i, _f in enumerate(dof_figs):
            pdf.savefig(_f, bbox_inches='tight')
            if _i == 0:
                plt.show()
            plt.close(_f)
    print(f'Wrote {len(dof_figs)} DOF page(s) to {output_pdf_dof_ordinal}')
else:
    print('(v-mode / DOF vs-ordinal skipped — OFC SVD unavailable)')


<a id='plots'></a>
## 6. Summary Heatmaps

For each `(bounce, comparison)` pair, produce three heatmaps over the
(k, j) grid:

* **Δ DZ heatmap** — colour = `comparison.median - reference.median`,
  cell text = `Δ\n±err`, diverging colormap.
* **Significance heatmap** — colour = `Δ / err`, capped at ±`sig_vlim`.
* **Pass / fail heatmap** — single colour for `(k, j)` cells that pass
  BOTH `|Δ| > pass_delta_threshold_um` and `|Δ/σ| > pass_nsigma_threshold`;
  the rest stay white.  Passing cells are annotated with `Δ\n±err`.

All three are saved to `output_pdf`.

In [ ]:
Path(output_dir).mkdir(parents=True, exist_ok=True)

n_pages = 0
with PdfPages(output_pdf) as pdf:
    for name, br in bounce_results.items():
        for comp_label, comp_block in br['comparisons'].items():
            deltas = comp_block['deltas']
            n_ref = br['reference_n']
            n_comp = comp_block['comp_n']
            descr = br['description']

            # 1. Δ heatmap with cell text Δ±err
            fig = plot_kj_heatmap(
                deltas, k_list, iZs,
                value_key='delta', err_key='err',
                title=(f'{name}: Δ DZ_kj = {comp_label} − {br["reference_label"]}\n'
                        f'{descr}  '
                        f'(n_ref={n_ref}, n_comp={n_comp})'),
                cbar_label='Δ DZ [μm]',
                cmap='RdBu_r', vlim=heatmap_vlim_um,
                value_fmt='{:+.3f}', err_fmt='±{:.3f}',
                cell_fontsize=heatmap_cell_fontsize)
            pdf.savefig(fig, bbox_inches='tight')
            if n_pages == 0:
                plt.show()
            plt.close(fig)
            n_pages += 1

            # 2. Significance heatmap (Δ / err)
            fig = plot_kj_heatmap(
                deltas, k_list, iZs,
                value_key='sig', err_key=None,
                title=(f'{name}: significance = Δ / σ  '
                        f'(capped ±{sig_vlim:g}σ)'),
                cbar_label='Δ / σ',
                cmap='RdBu_r', vlim=sig_vlim,
                value_fmt='{:+.1f}', err_fmt='',
                cell_fontsize=heatmap_cell_fontsize)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            n_pages += 1

            # 3. Binary "real shift" map: passes if |Δ| > delta_th AND
            #    |Δ/σ| > nsigma_th.  Single colour for passing cells.
            fig = plot_kj_pass_heatmap(
                deltas, k_list, iZs,
                nsigma_threshold=pass_nsigma_threshold,
                delta_threshold_um=pass_delta_threshold_um,
                sigma_only_threshold=pass_sigma_only_threshold,
                title=f'{name}: significant Δ DZ_kj cells (all nights)',
                cell_fontsize=heatmap_cell_fontsize)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)
            n_pages += 1

            # 4. Term inclusion = union of the combined cut AND each
            #    individual night's cut, so any term significant on a
            #    single night is shown (per-night pass heatmaps reveal why).
            dbn = comp_block.get('deltas_by_night', {})
            passing_set = passing_terms(
                deltas, pass_delta_threshold_um, pass_nsigma_threshold,
                sigma_only_th=pass_sigma_only_threshold)
            for _d in dbn.values():
                passing_set |= passing_terms(
                    _d, pass_delta_threshold_um, pass_nsigma_threshold,
                    sigma_only_th=pass_sigma_only_threshold)
            passing_kj = sorted(passing_set)

            # Per-night pass heatmaps (green boxes) — one page per night.
            for _night in sorted(dbn.keys()):
                fig = plot_kj_pass_heatmap(
                    dbn[_night], k_list, iZs,
                    nsigma_threshold=pass_nsigma_threshold,
                    delta_threshold_um=pass_delta_threshold_um,
                    sigma_only_threshold=pass_sigma_only_threshold,
                    title=f'{name}: significant Δ DZ_kj cells — night {_night}',
                    cell_fontsize=heatmap_cell_fontsize)
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)
                n_pages += 1

            # Per-night cross-comparison scatter — one PAGE per night pair.
            xfigs = plot_night_cross_scatter(
                dbn, passing_kj,
                title_root=(f'{name}: per-night Δ DZ_kj cross-comparison  '
                            f'({comp_label} − {br["reference_label"]})'),
                zoom_lim_um=cross_scatter_zoom_um)
            if xfigs:
                for fig in xfigs:
                    pdf.savefig(fig, bbox_inches='tight')
                    plt.close(fig)
                    n_pages += 1
            else:
                print(f'  ({name}/{comp_label}: night cross-scatter skipped '
                      f'— need >= 2 nights and >= 1 passing term)')

print(f'Wrote {n_pages} pages to {output_pdf}')

<a id='dof-scatter'></a>
## 9. DOF night-A vs night-B scatter

For the elevation bounce (T720) and rotator bounce (T724), the physical
DOF Δ from night A vs night B on a single 5-panel page: Cam & M2 hex
pistons; hex decenters; hex tip/tilts; M1M3 bending (20); M2 bending
(20) — all 50 DOF, errorbars from the paired-Δ per night.  Streamed to
`study_bounce_dof_night_scatter.pdf`.

In [ ]:
# DOF night-vs-night 5-panel scatter (T720 + T724)
if _svd_ok and C_all is not None:
    n_dof_pages = 0
    with PdfPages(output_pdf_dof_scatter) as pdf:
        for name, br in bounce_results.items():
            for comp_label, comp_block in br['comparisons'].items():
                dbn = comp_block.get('dof_deltas_by_night', {})
                if len(dbn) < 2:
                    print(f'  ({name}/{comp_label}: DOF night-scatter '
                          f'skipped — need >= 2 qualifying nights)')
                    continue
                figs = plot_dof_night_scatter(
                    dbn, LABELS_50DOF, units=DOF_UNITS_50,
                    title_root=(f'{name}: DOF Δ ({comp_label} - '
                                f'{br["reference_label"]})'),
                    night_pair=comp_block.get('night_pair'))
                for fig in figs:
                    pdf.savefig(fig, bbox_inches='tight')
                    if n_dof_pages == 0:
                        plt.show()
                    plt.close(fig)
                    n_dof_pages += 1
    print(f'Wrote {n_dof_pages} DOF night-scatter page(s) to '
          f'{output_pdf_dof_scatter}')
else:
    print('(DOF night-scatter skipped — OFC SVD unavailable)')


<a id='output'></a>
## 7. Output Tables

Long-format parquet with one row per `(bounce, comparison, k, j)`,
carrying both the per-setting medians/RMS and the propagated Δ.

In [ ]:
Path(output_dir).mkdir(parents=True, exist_ok=True)
df_kj.to_parquet(output_table_path)
print(f'Saved: {output_table_path}\n  {len(df_kj)} rows x '
      f'{len(df_kj.columns)} columns')

with pd.option_context('display.max_rows', 50,
                       'display.float_format', '{:+.4f}'.format):
    display(df_kj.sort_values(['bounce', 'comparison', 'j', 'k'])
                  .reset_index(drop=True)
                  .head(50))